# Random Forest Benchmark — Credit Card Default Prediction

**Notebook 03** in the credit default prediction pipeline.

This notebook implements and evaluates a **hyperparameter-tuned Random Forest** classifier as the baseline benchmark for comparison against the from-scratch Transformer model. Both models share the same preprocessing pipeline (`src/data_preprocessing.py`) to ensure a fair head-to-head evaluation.

---

| Property | Value |
|:---|:---|
| **Pipeline position** | Step 5 of 6 (Random Forest Benchmark) |
| **Data** | UCI Credit Card Default Dataset (30,000 clients) |
| **Features** | 23 original + 22 engineered = 45 total |
| **Primary metric** | AUC-ROC (class-imbalance robust, threshold-invariant) |
| **Secondary metric** | F1-score at optimised threshold |
| **Tuning strategy** | RandomizedSearchCV (60 iter × 5-fold CV) |

## Contents

1. [Setup & Configuration](#1.-Setup-&-Configuration)
2. [Data Pipeline Integration](#2.-Data-Pipeline-Integration)
3. [Baseline Model](#3.-Baseline-Model)
4. [Hyperparameter Tuning](#4.-Hyperparameter-Tuning)
5. [Model Evaluation](#5.-Model-Evaluation)
6. [Cross-Validation](#6.-Cross-Validation)
7. [Feature Importance Analysis](#7.-Feature-Importance-Analysis)
8. [Threshold Optimisation](#8.-Threshold-Optimisation)
9. [Results Summary](#9.-Results-Summary)

---

## 1. Setup & Configuration

Import the integrated Random Forest module and shared preprocessing pipeline. All data transformations (schema normalisation, categorical cleaning, feature engineering, stratified splitting) are handled by `src/data_preprocessing.py` to guarantee consistency with the Transformer pipeline.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Add src/ to path for module imports
sys.path.insert(0, str(Path("../src")))

from data_preprocessing import (
    load_raw_data, normalise_schema, clean_categoricals,
    engineer_features, split_data, TARGET_COL, RANDOM_SEED,
)
from random_forest import (
    train_baseline, tune_hyperparameters, evaluate_model,
    get_classification_report, cross_validate_model,
    compute_feature_importance, optimize_threshold,
    plot_roc_pr_curves, plot_confusion_matrix,
    plot_feature_importance, plot_threshold_analysis,
    plot_tuning_analysis, export_results,
    set_rf_style, PALETTE, TUNING_GRID,
)

# Apply publication-quality style
set_rf_style()
%matplotlib inline

# Output directories
FIGURE_DIR = "../figures"
RESULTS_DIR = "../results"
Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULTS_DIR).mkdir(exist_ok=True)

print("Setup complete.")

---

## 2. Data Pipeline Integration

We reuse the **shared preprocessing pipeline** to ensure identical data transformations across the RF and Transformer models. This is critical for a fair comparison — both models see exactly the same features, the same engineered variables, and the same train/val/test rows.

The pipeline performs:
- **Schema normalisation** — canonical column names, PAY_1 → PAY_0
- **Categorical cleaning** — undocumented EDUCATION/MARRIAGE codes merged into "Others"
- **Feature engineering** — 22 derived features (utilisation ratios, repayment ratios, delinquency aggregates, temporal dynamics)
- **Stratified splitting** — 70/15/15 preserving the 22.1% default rate

In [ ]:
# Load and preprocess using the shared pipeline
df = load_raw_data()
df = normalise_schema(df)
df = clean_categoricals(df)

print(f"\nClean dataset: {df.shape[0]:,} rows \u00d7 {df.shape[1]} columns")
print(f"Default rate:  {df['DEFAULT'].mean():.1%} ({df['DEFAULT'].sum():,} defaults)")
print(f"Class ratio:   {(1 - df['DEFAULT'].mean()) / df['DEFAULT'].mean():.1f}:1 imbalance")

### 2.1 Feature Engineering

The RF benefits from **explicit interaction features** that encode domain knowledge. Trees can split on these derived variables directly, whereas the Transformer must learn these patterns from raw features via self-attention.

In [ ]:
# Engineer 22+ derived features
df = engineer_features(df)

n_original = 23
n_total = df.shape[1] - 1  # exclude target
n_engineered = n_total - n_original

print(f"\nOriginal features:   {n_original}")
print(f"Engineered features: {n_engineered}")
print(f"Total features:      {n_total}")

# List the engineered feature groups
from data_preprocessing import ALL_FEATURE_COLS
eng_cols = [c for c in df.columns if c not in ALL_FEATURE_COLS and c != TARGET_COL]

groups = {
    "Utilisation ratios": [c for c in eng_cols if c.startswith("UTIL_RATIO")],
    "Repayment ratios": [c for c in eng_cols if c.startswith("REPAY_RATIO")],
    "Delinquency agg.": [c for c in eng_cols if c in [
        "N_MONTHS_DELAYED", "MAX_DELAY", "RECENT_DELAY", "DELAY_TREND", "N_MONTHS_NO_USE"
    ]],
    "Bill dynamics": [c for c in eng_cols if c in ["BILL_SLOPE", "AVG_UTIL_RATIO"]],
    "Payment dynamics": [c for c in eng_cols if c in ["AVG_PAY_AMT", "PAY_AMT_VOLATILITY"]],
    "Balance totals": [c for c in eng_cols if c in ["TOTAL_BILL", "TOTAL_PAY", "PAY_BILL_RATIO_TOTAL"]],
}

for group, cols in groups.items():
    print(f"  {group:20s} ({len(cols):2d}): {', '.join(cols)}")

### 2.2 Data Splitting

In [ ]:
# Stratified 70/15/15 split
train_df, val_df, test_df = split_data(df)

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]
X_val   = val_df.drop(columns=[TARGET_COL])
y_val   = val_df[TARGET_COL]
X_test  = test_df.drop(columns=[TARGET_COL])
y_test  = test_df[TARGET_COL]

# Verify class balance preservation
split_info = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Rows": [len(X_train), len(X_val), len(X_test)],
    "Features": [X_train.shape[1], X_val.shape[1], X_test.shape[1]],
    "Default Rate": [f"{y_train.mean():.2%}", f"{y_val.mean():.2%}", f"{y_test.mean():.2%}"],
    "% of Total": [f"{len(X_train)/len(df):.0%}", f"{len(X_val)/len(df):.0%}", f"{len(X_test)/len(df):.0%}"],
})
display(split_info.style.set_properties(**{"text-align": "center"}).hide(axis="index"))

---

## 3. Baseline Model

First, we train a Random Forest with **default hyperparameters** (100 estimators, no depth limit, no class weighting) to establish a performance floor. This baseline lets us quantify exactly how much benefit hyperparameter tuning provides.

In [ ]:
# Train baseline RF (default sklearn parameters)
baseline_model, baseline_time = train_baseline(X_train, y_train)
baseline_metrics, baseline_pred, baseline_prob = evaluate_model(
    baseline_model, X_test, y_test, "test_baseline"
)
baseline_metrics["model"] = "RF_baseline"
baseline_metrics["train_time_s"] = round(baseline_time, 2)

In [ ]:
# Classification report
print(get_classification_report(y_test, baseline_pred))

# Metrics summary
baseline_df = pd.DataFrame([baseline_metrics]).drop(columns=["split", "model"]).T
baseline_df.columns = ["Baseline RF"]
display(baseline_df.style.format("{:.4f}"))

---

## 4. Hyperparameter Tuning

We use **RandomizedSearchCV** with 60 random parameter combinations and 5-fold stratified cross-validation (300 total fits). Randomised search is preferred over exhaustive grid search because:

1. The full grid has 4 × 5 × 3 × 3 × 2 × 3 = **1,080 combinations** — exhaustive search would require 5,400 fits
2. Bergstra & Bengio (2012) showed that random search finds near-optimal configurations with far fewer evaluations, particularly when some hyperparameters matter more than others

### Search Space

| Parameter | Values | Rationale |
|:---|:---|:---|
| `n_estimators` | 100, 200, 300, 500 | More trees reduce variance; diminishing returns past ~300 |
| `max_depth` | 5, 10, 15, 20, None | Controls bias–variance trade-off; None = fully grown |
| `min_samples_split` | 2, 5, 10 | Regularisation against overfitting |
| `min_samples_leaf` | 1, 2, 4 | Smooths decision boundaries at leaf level |
| `max_features` | sqrt, log2 | Feature subsampling for tree decorrelation |
| `class_weight` | None, balanced, balanced_subsample | Addresses 3.5:1 class imbalance |

**Scoring:** AUC-ROC (threshold-invariant, robust to class imbalance).

In [ ]:
# Run RandomizedSearchCV (60 combinations x 5 folds = 300 fits)
searcher = tune_hyperparameters(X_train, y_train, n_iter=60, n_cv_folds=5)
best_model = searcher.best_estimator_

In [ ]:
# Best hyperparameters
params_df = pd.DataFrame(
    [(k, str(v)) for k, v in searcher.best_params_.items()],
    columns=["Parameter", "Best Value"],
)
display(params_df.style.hide(axis="index"))

print(f"\nBest CV AUC-ROC: {searcher.best_score_:.4f}")

### 4.1 Tuning Analysis

Four diagnostic plots to understand the hyperparameter landscape:

1. **n_estimators vs AUC** — ensemble size effect at different tree depths
2. **max_depth effect** — tree complexity vs generalisation
3. **class_weight effect** — impact of imbalance correction strategies
4. **Train vs CV AUC** — overfitting diagnostic for the top 20 configurations

In [ ]:
fig = plot_tuning_analysis(searcher, FIGURE_DIR)
plt.show()

---

## 5. Model Evaluation

Evaluate the tuned model on both the **validation set** (used for threshold tuning in Section 8) and the **held-out test set** (final, unbiased performance estimate).

In [ ]:
# Validation set evaluation
val_metrics, val_pred, y_val_prob = evaluate_model(
    best_model, X_val, y_val, "validation"
)

In [ ]:
# Test set evaluation
test_metrics, y_test_pred, y_test_prob = evaluate_model(
    best_model, X_test, y_test, "test"
)
test_metrics["model"] = "RF_tuned"

print("\n" + get_classification_report(y_test, y_test_pred))

### 5.1 ROC & Precision–Recall Curves

- **ROC curve** — plots true positive rate vs false positive rate across all thresholds. AUC-ROC = 1.0 is perfect, 0.5 is random.
- **Precision–Recall curve** — more informative than ROC under class imbalance (Saito & Rehmsmeier, 2015). The no-skill baseline equals the prevalence (22.1%).

In [ ]:
fig = plot_roc_pr_curves(y_test, y_test_prob, FIGURE_DIR)
plt.show()

---

## 6. Cross-Validation

5-fold stratified CV provides a robust performance estimate with confidence intervals. We run this on the **training set** using the tuned model's hyperparameters to assess stability — low standard deviation indicates that the model's performance is not sensitive to the particular train/val partition.

In [ ]:
cv_df = cross_validate_model(best_model, X_train, y_train, n_folds=5)

In [ ]:
# Format CV results
cv_display = cv_df.copy()
cv_display["summary"] = cv_display.apply(
    lambda r: f"{r['mean']:.4f} \u00b1 {r['std']:.4f}", axis=1
)
cv_display["range"] = cv_display.apply(
    lambda r: f"[{r['min']:.4f}, {r['max']:.4f}]", axis=1
)
display(
    cv_display[["metric", "summary", "range"]]
    .rename(columns={"metric": "Metric", "summary": "Mean \u00b1 Std", "range": "Range"})
    .style.hide(axis="index")
)

---

## 7. Feature Importance Analysis

We compute two complementary importance measures:

| Method | Type | Measures | Limitation |
|:---|:---|:---|:---|
| **Gini (MDI)** | Built-in | Total decrease in node impurity across all trees | Biased towards high-cardinality and correlated features |
| **Permutation** | Model-agnostic | Decrease in AUC-ROC when a feature is randomly shuffled | Accounts for feature correlations; more reliable for inference |

Permutation importance is the more trustworthy measure for feature selection decisions. Gini importance is useful for understanding what the trees are actually splitting on.

In [ ]:
importance_df = compute_feature_importance(best_model, X_test, y_test, top_n=20)

In [ ]:
fig = plot_feature_importance(importance_df, top_n=20, save_dir=FIGURE_DIR)
plt.show()

In [ ]:
# Top 15 features by permutation importance
top15 = importance_df.head(15)[["feature", "gini_importance", "perm_importance", "perm_std"]].copy()
top15.index = range(1, len(top15) + 1)
top15.index.name = "Rank"
top15.columns = ["Feature", "Gini", "Permutation", "Perm Std"]
display(top15.style.format({"Gini": "{:.4f}", "Permutation": "{:.4f}", "Perm Std": "{:.4f}"}))

### Key Observations

- **PAY features dominate** both importance measures, consistent with the EDA finding that repayment status is the strongest predictor of default
- **Engineered delinquency aggregates** (e.g., `N_MONTHS_DELAYED`, `MAX_DELAY`) rank highly, confirming that the feature engineering captures meaningful risk signals
- **BILL_AMT features show low permutation importance** despite moderate Gini importance — this is the classic MDI bias towards correlated features (BILL_AMT1–6 have r > 0.9)
- The discrepancy between Gini and permutation rankings highlights why relying on MDI alone can be misleading

---

## 8. Threshold Optimisation

The default classification threshold (0.50) assumes equal misclassification costs. In credit default prediction, **false negatives are typically more costly** than false positives — approving a client who will default is worse than declining a creditworthy client.

We sweep thresholds from 0.10 to 0.90 on the **validation set** (never touching the test set) and select the threshold that maximises the F1 score — a balanced measure of precision and recall.

In [ ]:
best_threshold, threshold_df = optimize_threshold(y_val, y_val_prob)

In [ ]:
fig = plot_threshold_analysis(threshold_df, best_threshold, FIGURE_DIR)
plt.show()

In [ ]:
# Re-evaluate test set at the optimal threshold
from sklearn.metrics import f1_score

test_pred_opt = (y_test_prob >= best_threshold).astype(int)
test_f1_default = f1_score(y_test, y_test_pred)
test_f1_optimal = f1_score(y_test, test_pred_opt)

print(f"Test F1 at default threshold (0.50):   {test_f1_default:.4f}")
print(f"Test F1 at optimal threshold ({best_threshold:.2f}):  {test_f1_optimal:.4f}")
print(f"Improvement: +{(test_f1_optimal - test_f1_default):.4f}")

### 8.1 Confusion Matrix at Optimal Threshold

The confusion matrix below shows the classification outcomes at the optimised threshold. Lowering the threshold from 0.50 increases recall (catching more defaulters) at the cost of some precision (more false alarms).

In [ ]:
fig = plot_confusion_matrix(y_test, y_test_prob, best_threshold, FIGURE_DIR)
plt.show()

---

## 9. Results Summary

In [ ]:
# Side-by-side comparison: Baseline vs Tuned
comparison = pd.DataFrame({
    "Metric": ["AUC-ROC", "F1 (t=0.50)", "Precision", "Recall",
               "Avg Precision", "Accuracy"],
    "Baseline RF": [
        baseline_metrics["auc_roc"], baseline_metrics["f1"],
        baseline_metrics["precision"], baseline_metrics["recall"],
        baseline_metrics["avg_precision"], baseline_metrics["accuracy"],
    ],
    "Tuned RF": [
        test_metrics["auc_roc"], test_metrics["f1"],
        test_metrics["precision"], test_metrics["recall"],
        test_metrics["avg_precision"], test_metrics["accuracy"],
    ],
})
comparison["Delta"] = comparison["Tuned RF"] - comparison["Baseline RF"]

display(
    comparison.style
    .format({"Baseline RF": "{:.4f}", "Tuned RF": "{:.4f}", "Delta": "+{:.4f}"})
    .hide(axis="index")
    .set_caption("Baseline vs Tuned Random Forest (Test Set)")
)

In [ ]:
# Export all results
export_results(
    baseline_metrics, test_metrics, cv_df, importance_df,
    searcher.best_params_, best_threshold, RESULTS_DIR,
)

### Key Takeaways

1. **Hyperparameter tuning provides modest but consistent improvement** over the default RF, primarily through better regularisation (max_depth, min_samples_leaf) and class-weight adjustment.

2. **PAY status features are the dominant predictors**, with PAY_0 (most recent repayment status) being the single strongest feature by both Gini and permutation importance. This aligns with the EDA finding that recent delinquency is the most informative signal.

3. **Engineered features add value** — delinquency aggregates and utilisation ratios rank among the top features, confirming that domain-informed feature engineering benefits the RF.

4. **Threshold optimisation shifts the precision–recall trade-off** — the optimal threshold (typically below 0.50) increases recall at a small precision cost, reflecting the asymmetric cost structure in credit risk.

5. **Cross-validation shows stable performance** with low standard deviation, indicating the model generalises well and is not overly sensitive to the particular data split.

---

### Implications for Transformer Comparison

The RF benchmark establishes a strong baseline. The Transformer model must demonstrate:
- **Competitive or superior AUC-ROC** on the same test set
- **Ability to capture temporal patterns** that the RF (with engineered features) cannot
- **Interpretability via attention weights** — which features and time steps the model attends to

The key question is whether self-attention over the raw 6-month payment sequences can match or exceed the RF operating on hand-crafted temporal features.